In [ ]:
!pip install transformers pandas numpy torch pyarrow fastparquet tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.1 MB/s eta 0:00:0000:010:01


In [ ]:
import os
import pandas as pd
import numpy as np
import re
import torch
from transformers import BertTokenizer
from tqdm import tqdm

In [ ]:
from google.colab import drive

# 1. Mount Drive (Essential for accessing your saved data.zip)
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set your data directory for Colab
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/data/" 
FILENAME = "final_dataset_uncleaned.csv" # UPDATE THIS to your actual filename

file_path = f"{DATA_DIR}/{FILENAME}"

print(f"Loading data from {file_path}...")
df = pd.read_csv(file_path, low_memory=False)
# "", ""
# Ensure the dataset has the required columns
# If your columns are named differently (e.g., 'content' and 'is_ai'), rename them here:
df = df.rename(columns={'content': 'full_post', 'is_ai': 'Label'})

# Drop any rows with missing text or labels right away
df = df.dropna(subset=['full_post', 'Label']).reset_index(drop=True)

print(f"Total rows loaded: {len(df)}")
print(df['Label'].value_counts())

Loading data from /content/drive/MyDrive/Colab Notebooks/data//final_dataset_uncleaned.csv...
Total rows loaded: 563089
Label
0    290251
1    272838
Name: count, dtype: int64


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove extra whitespace but keep single spaces
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# Apply cleaning with a progress bar
tqdm.pandas(desc="Cleaning Text")
df['full_post_clean'] = df['full_post'].progress_apply(clean_text)

# Drop rows that ended up being too short or empty after cleaning
df = df[df['full_post_clean'].str.len() >= 5].reset_index(drop=True)

print(f"Dataset shape after cleaning: {df.shape}")

# EXPORT 1: Save the cleaned tabular data for your AI engineer directly back to Drive
tabular_output_path = f"{DATA_DIR}/bert_cleaned_dataset.parquet"
df.to_parquet(tabular_output_path, index=False)
print(f"Saved cleaned tabular dataset to {tabular_output_path}")

Cleaning Text: 100%|██████████| 563089/563089 [00:23<00:00, 23716.93it/s]


Dataset shape after cleaning: (562618, 3)
Saved cleaned tabular dataset to /content/drive/MyDrive/Colab Notebooks/data//bert_cleaned_dataset.parquet


In [7]:
print("Loading BERT Tokenizer...")
# Ensure .from_pretrained() is used so it becomes an instance, not a class object
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

texts = df['full_post_clean'].values
labels = df['Label'].values

input_ids = []
attention_masks = []

print("Tokenizing 590,000+ texts... this will take a few minutes.")
for text in tqdm(texts, desc="Tokenization Progress"):
    # Simply call the tokenizer directly instead of using .encode_plus()
    encoded_dict = tokenizer(
        text,                      
        add_special_tokens = True, 
        max_length = 256,          
        padding = 'max_length',    # Updated to the modern parameter name
        truncation = True,
        return_attention_mask = True,   
        return_tensors = 'pt'     
    )
    
    input_ids.append(encoded_dict['input_ids'])
    attention_masks.append(encoded_dict['attention_mask'])

print("Concatenating tensors into memory...")
# Convert the lists of tensors into a single massive PyTorch tensor
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels, dtype=torch.long)

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Masks shape: {attention_masks.shape}")
print(f"Labels shape: {labels.shape}")

Loading BERT Tokenizer...
Tokenizing 590,000+ texts... this will take a few minutes.


Tokenization Progress: 100%|██████████| 562618/562618 [10:51<00:00, 864.03it/s] 


Concatenating tensors into memory...
Input IDs shape: torch.Size([562618, 256])
Attention Masks shape: torch.Size([562618, 256])
Labels shape: torch.Size([562618])


In [8]:
# EXPORT 2: Save the fully tokenized tensors for PyTorch directly back to Drive
tensor_output_path = f"{DATA_DIR}/bert_tokenized_tensors.pt"

print("Saving tensors to disk... do not close the notebook until finished.")
export_data = {
    'input_ids': input_ids,
    'attention_masks': attention_masks,
    'labels': labels
}

torch.save(export_data, tensor_output_path)
print(f"Saved tokenized PyTorch tensors to {tensor_output_path}")
print("Data export complete. The AI engineer can now load this .pt file from your Google Drive.")

Saving tensors to disk... do not close the notebook until finished.
Saved tokenized PyTorch tensors to /content/drive/MyDrive/Colab Notebooks/data//bert_tokenized_tensors.pt
Data export complete. The AI engineer can now load this .pt file from your Google Drive.
